In [1]:
from pyspark.sql import SparkSession
import getpass

In [2]:
username = getpass.getuser()

In [ ]:
spark = SparkSession.builder. \
config('spark.sql.warehouse.dir', f'/user/{username}/warehouse'). \
config("spark.shuffle.service.enabled", "false"). \
config('spark.dynamicAllocation.enabled', 'false'). \
config('spark.executor.instances', '5'). \
enableHiveSupport(). \
master('yarn'). \
appName('2530_caching'). \
getOrCreate()

In [4]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType, IntegerType

order_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("order_date", TimestampType()),
    StructField("customer_id", LongType()),
    StructField("order_status", StringType())
])

In [5]:
load_df = spark.read.format('csv').schema(order_schema).load('datasets/orders_1gb.csv')

In [6]:
#load_df.show(5)

In [7]:
load_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_status: string (nullable = true)



In [8]:
load_df.storageLevel

StorageLevel(False, False, False, False, 1)

In [9]:
#load_df.describe()

In [10]:
#load_df.summary()

In [11]:
#print(load_df.rdd.getNumPartitions())

In [12]:
# filtered_df = load_df.filter("order_status IN ('COMPLETE', 'CLOSED')")
filtered_df = load_df.filter("order_status IN ('COMPLETE')")

In [13]:
#filtered_df.show(10)

In [14]:
#print(filtered_df.rdd.getNumPartitions())

In [15]:
cached_df = filtered_df.cache()

In [16]:
unique_cols = cached_df.dropDuplicates(['order_id', 'order_status'])

In [17]:
import time
start_time = time.time()

unique_cols.show(5)
print(f"Time to process cache: {time.time() - start_time} seconds")

+--------+-------------------+-----------+------------+
|order_id|         order_date|customer_id|order_status|
+--------+-------------------+-----------+------------+
|   16098|2013-11-04 00:00:00|       5939|    COMPLETE|
|   16368|2013-11-05 00:00:00|      10739|    COMPLETE|
|   16942|2013-11-07 00:00:00|      10051|    COMPLETE|
|   17158|2013-11-09 00:00:00|       1595|    COMPLETE|
|   19276|2013-11-22 00:00:00|       8919|    COMPLETE|
+--------+-------------------+-----------+------------+
only showing top 5 rows

Time to process cache: 69.6784098148346 seconds


In [18]:
#cached_df.unpersist()

In [19]:
#spark.sql("clear cache")

In [20]:
#spark.stop()

In [23]:
from pyspark.sql.functions import count

start_time = time.time()
output_df = cached_df.groupBy("customer_id").agg(count('order_id').alias("order_count")).sort('order_count', ascending=False)
output_df.show()
print(f"Time to process groupBy which is a wide and compute heavy operation is: {time.time() - start_time} seconds")

+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|       9337|       3750|
|       7802|       3375|
|       3710|       3375|
|        749|       3375|
|       2469|       3000|
|       7910|       3000|
|       5186|       3000|
|        221|       3000|
|      11061|       3000|
|       5283|       3000|
|       8861|       2625|
|       8314|       2625|
|       8930|       2625|
|        363|       2625|
|       5600|       2625|
|       4116|       2625|
|       3763|       2625|
|      12242|       2625|
|       7981|       2625|
|       9325|       2625|
+-----------+-----------+
only showing top 20 rows

Time to process groupBy which is a wide and compute heavy operation is: 0.7882602214813232 seconds


### Personal_note: Refer spark ui in folder named "caching_1"